## **Combinación de predicciones**

En este notebook se proponen diferentes alternativas para combinar los resultados obtenidos por el modelo sobre datos tabulares, el modelo de imágenes y el modelo de texto.

* **Probabilidades promedio:** Se calculan las probabilidades promedio de cada clase en base a los 3 modelos y se decide predice la clase que tiene mayor probabilidad.
* **Predicción combinación lineal:** Se busca la mejor combinación lineal de las probabilidades (sujeto a que la suma de los coeficientes utilizados sea igual a 1).

#### **Librerias a utilizar**

In [1]:
import pandas as pd

#Funciones auxiliares sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold #Split y cross Validation
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score #Metricas
from sklearn.utils import shuffle

#Visualizacióon
from plotly import express as px

#Plot de matriz de confusion normalizada en actuals
from utils import plot_confusion_matrix

#### **Resultados LightGBM**

In [2]:
pred_lightgbm = pd.read_csv("C:/Users/FBorbiconi/Downloads/pred_lightgbm.csv")

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(pred_lightgbm["AdoptionSpeed"], pred_lightgbm["Pred_DT"]))

cohen_kappa_score(pred_lightgbm["AdoptionSpeed"], pred_lightgbm["Pred_DT"], weights = 'quadratic')

0.4081599097602615

#### **Resultados imágenes (ResNet50)**

In [6]:
pred_imagenes = pd.read_csv("C:/Users/FBorbiconi/Downloads/pred_imagenes.csv")

# Combino con el dataset de lightGBM
Resultados = pd.merge(pred_lightgbm, pred_imagenes, on = "PetID", how="left")

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(Resultados[Resultados["Im_Pred"].notna()]["AdoptionSpeed"], Resultados[Resultados["Im_Pred"].notna()]["Im_Pred"]))

cohen_kappa_score(Resultados[Resultados["Im_Pred"].notna()]["AdoptionSpeed"], Resultados[Resultados["Im_Pred"].notna()]["Im_Pred"], weights = 'quadratic')

0.3195301757630282

#### **Resultados texto (DistilBERT)**

In [7]:
pred_txt = pd.read_csv("C:/Users/FBorbiconi/Downloads/pred_txt_traduccion.csv")

# Combino los resultados con el resto de predicciones
Resultados = pd.merge(Resultados, pred_txt, on = "PetID", how="left")

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(Resultados[Resultados["txt_Pred"].notna()]["AdoptionSpeed"], Resultados[Resultados["txt_Pred"].notna()]["txt_Pred"]))

cohen_kappa_score(Resultados[Resultados["txt_Pred"].notna()]["AdoptionSpeed"], Resultados[Resultados["txt_Pred"].notna()]["txt_Pred"], weights = 'quadratic')

0.23590321729750197

#### **Combinación de Resultados**

In [8]:
# Se reemplazan los NA de probabilidades por 0
Resultados.loc[:, ~Resultados.columns.isin(["Im_pred", "txt_Pred"])] = Resultados.loc[:, ~Resultados.columns.isin(["Im_pred", "txt_Pred"])].fillna(0)


##### **Promedio de probabilidades**

In [9]:
# Calcular los promedios y crear 5 nuevas columnas
for i in range(5):
    Resultados[f'Promedio_Clase_{i}'] = (Resultados[f'DT_Clase_{i}'] + Resultados[f'Im_Clase_{i}'] + Resultados[f'txt_Clase_{i}']) / 3

# Realizar la prediccion con las 5 columnas promedio
columnas_promedio = [f'Promedio_Clase_{i}' for i in range(5)]

Resultados['Pred_Promedio'] = Resultados[columnas_promedio].idxmax(axis=1)

Resultados['Pred_Promedio'] = Resultados['Pred_Promedio'].str.split('_').str.get(-1).astype(int)

#Muestro matriz de confusion y kappa
display(plot_confusion_matrix(Resultados["AdoptionSpeed"], Resultados["Pred_Promedio"]))

cohen_kappa_score(Resultados["AdoptionSpeed"], Resultados["Pred_Promedio"], weights = 'quadratic')

0.2983914399665635

##### **Búsqueda de la mejor combinación**

In [26]:
import itertools

# Valores permitidos
valores = [round(i * 0.01, 1) for i in range(0, 101)]  

# Generar todas las combinaciones con repetición de 3 valores
combinaciones = [
    comb for comb in itertools.product(valores, repeat=3)
    if round(sum(comb), 1) == 1.0
]

max_kappa = 0

for c in combinaciones:
    # Calcular la combinación y crear 5 nuevas columnas
    for i in range(5):
        Resultados[f'Combinacion_Clase_{i}'] = (Resultados[f'DT_Clase_{i}'] * c[0] + Resultados[f'Im_Clase_{i}'] * c[1] + Resultados[f'txt_Clase_{i}'] * c[2])

    # Realizar la prediccion con las 5 columnas combinadas
    columnas_combinacion = [f'Combinacion_Clase_{i}' for i in range(5)]

    Resultados['Pred_Combinacion'] = Resultados[columnas_combinacion].idxmax(axis=1)

    Resultados['Pred_Combinacion'] = Resultados['Pred_Combinacion'].str.split('_').str.get(-1).astype(int)

    #Calculo matriz de confusion y kappa
    matriz = plot_confusion_matrix(Resultados["AdoptionSpeed"], Resultados["Pred_Combinacion"])

    kappa = cohen_kappa_score(Resultados["AdoptionSpeed"], Resultados["Pred_Combinacion"], weights = 'quadratic')

    if kappa > max_kappa:
        max_kappa = kappa
        max_matrix = matriz
        max_comb = c

display(max_matrix)
print(f'La mejor combinación es {max_comb} con un kappa de {max_kappa}')

La mejor combinación es (0.5, 0.4, 0.1) con un kappa de 0.44429944969262314
